<a href="https://colab.research.google.com/github/Optimus0205/Data_Science_Project/blob/main/Model_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [2]:
fs_df=pd.read_csv('/content/post_feature_selection.csv')
fs_df.head()

,Unnamed: 0,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,study room,servant room,store room,pooja room,others,furnishing_type,luxury_category,floor_category
0,0,flat,sohna road,0.87,2,2,3+,Relatively New,1131.0,1,0,0,0,0,0,MODERATE,Mid Floor
1,1,flat,sector 69,2.00,4,4,2,Relatively New,1889.0,0,0,0,0,0,0,HIGH,Low Floor
2,2,flat,sector 47,2.40,3,3,3+,Moderately Old,1888.0,0,1,0,0,0,1,MODERATE,Mid Floor
3,3,flat,sector 83,1.75,3,3,3,Relatively New,1600.0,0,1,0,0,0,1,HIGH,Mid Floor
4,4,flat,sector 82,0.63,2,2,3+,Relatively New,1026.0,1,0,0,0,0,0,MODERATE,Low Floor


In [3]:
fs_df.drop(columns=['Unnamed: 0', 'study room', 'pooja room', 'others'], inplace=True)
fs_df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sohna road,0.87,2,2,3+,Relatively New,1131.0,0,0,0,MODERATE,Mid Floor
1,flat,sector 69,2.00,4,4,2,Relatively New,1889.0,0,0,0,HIGH,Low Floor
2,flat,sector 47,2.40,3,3,3+,Moderately Old,1888.0,1,0,1,MODERATE,Mid Floor
3,flat,sector 83,1.75,3,3,3,Relatively New,1600.0,1,0,1,HIGH,Mid Floor
4,flat,sector 82,0.63,2,2,3+,Relatively New,1026.0,0,0,0,MODERATE,Low Floor


In [4]:
fs_df.to_csv('gurgaon_properties_post_feature_selection_v2.csv',index=False)

In [5]:
df=pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv')
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sohna road,0.87,2,2,3+,Relatively New,1131.0,0,0,0,MODERATE,Mid Floor
1,flat,sector 69,2.00,4,4,2,Relatively New,1889.0,0,0,0,HIGH,Low Floor
2,flat,sector 47,2.40,3,3,3+,Moderately Old,1888.0,1,0,1,MODERATE,Mid Floor
3,flat,sector 83,1.75,3,3,3,Relatively New,1600.0,1,0,1,HIGH,Mid Floor
4,flat,sector 82,0.63,2,2,3+,Relatively New,1026.0,0,0,0,MODERATE,Low Floor


In [6]:
df['furnishing_type'].value_counts()

,count
furnishing_type,
0,2131
1,962
2,175


In [7]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [8]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sohna road,0.87,2,2,3+,Relatively New,1131.0,0,0,unfurnished,MODERATE,Mid Floor
1,flat,sector 69,2.00,4,4,2,Relatively New,1889.0,0,0,unfurnished,HIGH,Low Floor
2,flat,sector 47,2.40,3,3,3+,Moderately Old,1888.0,1,0,semifurnished,MODERATE,Mid Floor
3,flat,sector 83,1.75,3,3,3,Relatively New,1600.0,1,0,semifurnished,HIGH,Mid Floor
4,flat,sector 82,0.63,2,2,3+,Relatively New,1026.0,0,0,unfurnished,MODERATE,Low Floor


In [9]:
X=df.drop(columns=['price'])
y=df['price']

In [10]:
# Applying log1p transformation on price
y_transformed=np.log1p(y)

## Ordinal Encoding

In [11]:
# categorical columns
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [12]:
# Trnsformer we have to aaply
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode)
    ],
    remainder='passthrough'
)

In [13]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [14]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [15]:
scores.mean(),scores.std()

(np.float64(0.731233043895259), np.float64(0.03580351826008316))

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [17]:
print(X_train[X_train['sector'] == 'sector 37'])

Empty DataFrame
Columns: [property_type, sector, bedRoom, bathroom, balcony, agePossession, built_up_area, servant room, store room, furnishing_type, luxury_category, floor_category]
Index: []


In [18]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [19]:
y_pred = pipeline.predict(X_test)

In [20]:
y_pred = np.expm1(y_pred)

In [21]:
mean_absolute_error(np.expm1(y_test),y_pred)
# not very good very huge deviation

0.9519800282472678

In [22]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [23]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [24]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [25]:
model_output

[['linear_reg', np.float64(0.731233043895259), 0.9519800282472678],
 ['svr', np.float64(0.7482720910234175), 0.8889281525000186],
 ['ridge', np.float64(0.7312387162892265), 0.9517571529972741],
 ['LASSO', np.float64(0.049118973920071765), 1.5474155209237663],
 ['decision tree', np.float64(0.7712525847018361), 0.7148796146345876],
 ['random forest', np.float64(0.8773685753228918), 0.544831613614495],
 ['extra trees', np.float64(0.8651620931545091), 0.6078368214010919],
 ['gradient boosting', np.float64(0.8699846126186335), 0.591165447984827],
 ['adaboost', np.float64(0.7533251788669633), 0.8292120324576195],
 ['mlp', np.float64(0.8054514928370688), 0.7408934827011764],
 ['xgboost', np.float64(0.8869321817857319), 0.5145095794326668]]

In [26]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [27]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.886932,0.514510
5,random forest,0.877369,0.544832
7,gradient boosting,0.869985,0.591165
6,extra trees,0.865162,0.607837
4,decision tree,0.771253,0.714880
9,mlp,0.805451,0.740893
8,adaboost,0.753325,0.829212
1,svr,0.748272,0.888928
2,ridge,0.731239,0.951757
0,linear_reg,0.731233,0.951980


## One Hot Encoding

In [28]:
# Creating a column transformer for preprocessing
ordinal_encode_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category']
onehot_encode_cols = ['sector','agePossession','furnishing_type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat_ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_encode_cols),
        ('cat_onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), onehot_encode_cols)
    ],
    remainder='passthrough'
)

In [29]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [30]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [31]:
scores.mean()

np.float64(0.8471849452965847)

In [32]:
scores.std()

np.float64(0.0337559644546749)

In [33]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [34]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat_ordinal',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['property_type', 'balcony',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat_onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sector', 'agePossession',
                                                   'furnishing_type'])])),
                ('regressor', LinearRegression())])

In [35]:
y_pred = pipeline.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [36]:
y_pred = np.expm1(y_pred)

In [37]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6875222206834645

In [38]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [39]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [40]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

In [41]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [42]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.882430,0.507442
10,xgboost,0.883051,0.541389
5,random forest,0.867725,0.571005
1,svr,0.877874,0.574442
9,mlp,0.863525,0.578427
7,gradient boosting,0.855578,0.637404
0,linear_reg,0.847185,0.687522
2,ridge,0.847639,0.691499
4,decision tree,0.780190,0.787024
8,adaboost,0.724010,0.856353


##OneHotEncoding With PCA

In [43]:
# Creating a column transformer for preprocessing
ordinal_encode_cols = ['property_type', 'balcony', 'luxury_category', 'floor_category']
onehot_encode_cols = ['sector','agePossession', 'furnishing_type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat_ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_encode_cols),
        ('cat_onehot', OneHotEncoder(drop='first',sparse_output=False, handle_unknown='ignore'), onehot_encode_cols)
    ],
    remainder='passthrough'
)

In [44]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [45]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [46]:
scores.mean()

np.float64(0.7572013866946478)

In [47]:
scores.std()

np.float64(0.03174155094104083)

In [48]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [49]:
  model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [50]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

In [51]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [52]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.828285,0.645678
10,xgboost,0.818152,0.648020
5,random forest,0.816229,0.676443
9,mlp,0.818670,0.703420
1,svr,0.818577,0.704502
7,gradient boosting,0.810624,0.733892
0,linear_reg,0.757201,0.882612
2,ridge,0.757231,0.882909
8,adaboost,0.680915,0.944651
4,decision tree,0.618733,1.055228


##Target Encoder

In [53]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.9 MB/s eta 0:00:00


In [54]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False, handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

In [55]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [56]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [57]:
scores.mean(),scores.std()

(np.float64(0.8174961719458474), np.float64(0.03379402575934344))

In [58]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [59]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [60]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [61]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [62]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.895425,0.476271
5,random forest,0.897226,0.490978
10,xgboost,0.897160,0.492341
7,gradient boosting,0.885028,0.558807
4,decision tree,0.810636,0.623646
9,mlp,0.842595,0.644274
8,adaboost,0.809766,0.743584
0,linear_reg,0.817496,0.780516
2,ridge,0.817530,0.780599
1,svr,0.764475,0.868103


##Hyperparameter Tuning

###Random Forest -> GridSearchCV

In [63]:
from sklearn.model_selection import GridSearchCV

In [70]:
param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['sqrt', 'log2', 1.0]
}

In [71]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False, handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

In [72]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [73]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [74]:
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)

In [75]:
search.fit(X, y_transformed)

Fitting 10 folds for each of 192 candidates, totalling 1920 fits


GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['bedRoom',
                                                                          'bathroom',
                                                                          'built_up_area',
                                                                          'servant '
                                                                          'room',
                                                                          'store '
                                                                          'room']),
                                                                        ('cat',
                                                                         OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                        unknown_value=-1),
                                                                         ['property_type',
                                                                          's...
                                                                                       handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         ['agePossession']),
                                                                        ('target_enc',
                                                                         TargetEncoder(),
                                                                         ['sector'])])),
                                       ('regressor', RandomForestRegressor())]),
             n_jobs=-1,
             param_grid={'regressor__max_depth': [None, 10, 20, 30],
                         'regressor__max_features': ['sqrt', 'log2', 1.0],
                         'regressor__max_samples': [0.1, 0.25, 0.5, 1.0],
                         'regressor__n_estimators': [50, 100, 200, 300]},
             scoring='r2', verbose=4)

In [76]:
final_pipe = search.best_estimator_

In [77]:
search.best_params_

{'regressor__max_depth': 20,
 'regressor__max_features': 1.0,
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 300}

In [78]:
search.best_score_

np.float64(0.8993236355708518)

In [79]:
final_pipe.fit(X,y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['agePossession']),
                                                 ('target_enc', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=20, max_samples=1.0,
                                       n_estimators=300))])

In [82]:
from sklearn.metrics import mean_absolute_error
import numpy as np

# Make predictions on the test set using the best estimator from GridSearchCV (final_pipe)
y_pred_tuned = final_pipe.predict(X_test)

# Inverse transform the predictions from log scale
y_pred_tuned_expm1 = np.expm1(y_pred_tuned)

# Calculate MAE for the GridSearchCV-tuned model
mae_tuned = mean_absolute_error(np.expm1(y_test), y_pred_tuned_expm1)

print(f"MAE after GridSearchCV Hyperparameter Tuning: {mae_tuned}")

MAE after GridSearchCV Hyperparameter Tuning: 0.18000857355974423


###Extra Tree Regressor -> RandomizedSearchCV

In [83]:
from sklearn.model_selection import RandomizedSearchCV

In [89]:
param_dist = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30, 50],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 'log2', 1.0] # 1.0 means using all features
}

In [90]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False, handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

In [91]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ExtraTreesRegressor())
])

In [92]:
random_search = RandomizedSearchCV(pipeline, param_dist, n_iter=20, cv=5, scoring='r2', n_jobs=-1, random_state=42)

In [93]:
random_search.fit(X, y_transformed)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('num',
                                                                               StandardScaler(),
                                                                               ['bedRoom',
                                                                                'bathroom',
                                                                                'built_up_area',
                                                                                'servant '
                                                                                'room',
                                                                                'store '
                                                                                'room']),
                                                                              ('cat',
                                                                               OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                              unknown_value=-1),
                                                                               ['property_type',
                                                                                'sector',
                                                                                'balcony',
                                                                                'agePossession',
                                                                                'furnis...
                                                                              ('target_enc',
                                                                               TargetEncoder(),
                                                                               ['sector'])])),
                                             ('regressor',
                                              ExtraTreesRegressor())]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'regressor__max_depth': [None, 10, 20,
                                                                 30, 50],
                                        'regressor__max_features': ['sqrt',
                                                                    'log2',
                                                                    1.0],
                                        'regressor__min_samples_leaf': [1, 2,
                                                                        4],
                                        'regressor__min_samples_split': [2, 5,
                                                                         10],
                                        'regressor__n_estimators': [50, 100,
                                                                    200, 300]},
                   random_state=42, scoring='r2')

In [94]:
final_pipe = search.best_estimator_

In [95]:
search.best_params_

{'regressor__max_depth': 20,
 'regressor__max_features': 1.0,
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 300}

In [96]:
search.best_score_

np.float64(0.8993236355708518)

In [97]:
final_pipe.fit(X,y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['agePossession']),
                                                 ('target_enc', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=20, max_samples=1.0,
                                       n_estimators=300))])

In [98]:
# Make predictions on the test set using the best estimator from RandomizedSearchCV
y_pred_randomized_search = final_pipe.predict(X_test)

# Inverse transform the predictions from log scale
y_pred_randomized_search_expm1 = np.expm1(y_pred_randomized_search)

# Calculate MAE
mae_randomized_search = mean_absolute_error(np.expm1(y_test), y_pred_randomized_search_expm1)

print(f"MAE after RandomizedSearchCV: {mae_randomized_search}")

MAE after RandomizedSearchCV: 0.1769433187682606


#Exporting the model

In [111]:
import category_encoders as ce
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Define columns for each specific encoding type to avoid overlap and redundancy
num_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']
ordinal_cat_cols = ['property_type', 'balcony', 'furnishing_type', 'luxury_category', 'floor_category'] # Excluded 'sector', 'agePossession'
onehot_cat_cols = ['agePossession'] # For OneHotEncoder
target_cat_cols = ['sector'] # For TargetEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat_ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cat_cols),
        ('cat_onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), onehot_cat_cols),
        ('target_enc', ce.TargetEncoder(), target_cat_cols)
    ],
    remainder='passthrough'
)

In [112]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=500))
])

In [113]:
pipeline.fit(X,y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat_ordinal',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['property_type', 'balcony',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat_onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['agePossession']),
                                                 ('target_enc', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor', RandomForestRegressor(n_estimators=500))])

In [114]:
import pickle

with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [115]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [116]:
X

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sohna road,2,2,3+,Relatively New,1131.0,0,0,unfurnished,MODERATE,Mid Floor
1,flat,sector 69,4,4,2,Relatively New,1889.0,0,0,unfurnished,HIGH,Low Floor
2,flat,sector 47,3,3,3+,Moderately Old,1888.0,1,0,semifurnished,MODERATE,Mid Floor
3,flat,sector 83,3,3,3,Relatively New,1600.0,1,0,semifurnished,HIGH,Mid Floor
4,flat,sector 82,2,2,3+,Relatively New,1026.0,0,0,unfurnished,MODERATE,Low Floor
...,...,...,...,...,...,...,...,...,...,...,...,...
3263,flat,sector 104,3,4,3+,Relatively New,2359.0,1,0,semifurnished,HIGH,Mid Floor
3264,flat,sector 71,2,2,1,Moderately Old,639.0,0,0,unfurnished,MODERATE,Mid Floor
3265,flat,sector 102,3,3,3,Relatively New,2200.0,1,0,unfurnished,LOW,High Floor
3266,flat,sector 92,3,4,3,Relatively New,1805.0,1,0,unfurnished,LOW,Mid Floor


##Trying out the predictions

In [117]:
X.columns

Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='object')

In [118]:
X.iloc[0].values

array(['flat', 'sohna road', np.int64(2), np.int64(2), '3+',
       'Relatively New', np.float64(1131.0), np.int64(0), np.int64(0),
       'unfurnished', 'MODERATE', 'Mid Floor'], dtype=object)

In [130]:
data = [['house', 'sector 102', 4, 3, '3+', 'New Property', 2750, 0, 0, 'unfurnished', 'LOW', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category']

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 102,4,3,3+,New Property,2750,0,0,unfurnished,LOW,Low Floor


In [131]:
np.expm1(pipeline.predict(one_df))

array([2.36852146])

In [121]:
X.dtypes

,0
property_type,object
sector,object
bedRoom,int64
bathroom,int64
balcony,object
agePossession,object
built_up_area,float64
servant room,int64
store room,int64
furnishing_type,object


In [122]:
sorted(X['sector'].unique().tolist())

['dwarka expressway',
 'gwal pahari',
 'manesar',
 'sector 1',
 'sector 102',
 'sector 103',
 'sector 104',
 'sector 105',
 'sector 106',
 'sector 107',
 'sector 108',
 'sector 109',
 'sector 10a',
 'sector 11',
 'sector 110',
 'sector 111',
 'sector 112',
 'sector 113',
 'sector 12',
 'sector 13',
 'sector 14',
 'sector 15',
 'sector 17',
 'sector 17a',
 'sector 17b',
 'sector 2',
 'sector 21',
 'sector 22',
 'sector 23',
 'sector 24',
 'sector 25',
 'sector 26',
 'sector 27',
 'sector 28',
 'sector 3',
 'sector 3 phase 2',
 'sector 3 phase 3 extension',
 'sector 30',
 'sector 31',
 'sector 33',
 'sector 36',
 'sector 36a',
 'sector 37',
 'sector 37c',
 'sector 37d',
 'sector 38',
 'sector 39',
 'sector 4',
 'sector 40',
 'sector 41',
 'sector 43',
 'sector 45',
 'sector 46',
 'sector 47',
 'sector 48',
 'sector 49',
 'sector 5',
 'sector 50',
 'sector 51',
 'sector 52',
 'sector 53',
 'sector 54',
 'sector 55',
 'sector 56',
 'sector 57',
 'sector 58',
 'sector 59',
 'sector 6',
 'se